# vLLM vs SGLang on a Colab GPU
Runtime → Change runtime type → **L4 GPU**, then Run all.

- Installs each engine in its own virtualenv (about 10 minutes).
- Runs a quick smoke test, then the full sweep.
- Total is roughly 1.5 hours, about 3 compute units.
- At the end it downloads `serving_results.zip`.

The notebook **stops** (raises an error) if an engine fails to install or the smoke test fails, so a broken setup does not burn an hour. If that happens, send the output of the failing cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('/content/lowbit-decode-gpu'):
    !git clone -q https://github.com/FredWen-S/lowbit-decode-gpu.git /content/lowbit-decode-gpu
%cd /content/lowbit-decode-gpu
!git pull -q && git log -1 --format='%h %s'

Each engine gets its own virtualenv, because their torch pins conflict. Versions are pinned to the ones of the first full run (vLLM 0.32.0, SGLang 0.5.23) so reruns stay comparable. The load generator runs in Colab's own Python.

In [ ]:
!pip -q install uv
!uv venv --clear --python 3.12 /content/venv-vllm
!uv pip install --python /content/venv-vllm/bin/python vllm==0.32.0 ninja 2>&1 | tail -25

In [ ]:
!uv venv --clear --python 3.12 /content/venv-sglang
!uv pip install --python /content/venv-sglang/bin/python 'sglang[all]==0.5.23' ninja 2>&1 | tail -25

In [ ]:
import os
import subprocess
for eng in ['vllm', 'sglang']:
    r = subprocess.run([f'/content/venv-{eng}/bin/python', '-c', f'import {eng}; print({eng}.__version__)'],
                       capture_output=True, text=True)
    print(eng, r.stdout.strip() or r.stderr[-2000:])
    assert r.returncode == 0, f'{eng} did not install; see the install cell above'
    assert os.path.exists(f'/content/venv-{eng}/bin/ninja'), f'ninja missing in venv-{eng} (flashinfer needs it)'

In [ ]:
from huggingface_hub import snapshot_download
MODEL = snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', local_dir='/content/models/qwen2.5-1.5b-instruct')
!pip -q install aiohttp
print(MODEL)

Smoke test with few requests. If a server fails to start, the last lines of its log are printed here.

In [ ]:
!rm -f /content/SMOKE_FAILED
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --configs default nocache chunk512 chunk4096 --quick --out results/serving_results_quick || touch /content/SMOKE_FAILED
import os
assert not os.path.exists('/content/SMOKE_FAILED'), 'smoke test failed; see the output above'

Full run (about 50 minutes).

In [ ]:
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --configs default nocache chunk512 chunk4096 --out results/serving_results

In [ ]:
!zip -qr /content/serving_results.zip results/serving_results results/serving_results_quick
from google.colab import files
files.download('/content/serving_results.zip')